# Pandas確認課題

このPandas確認問題は、データサイエンス100本ノックの問題で最低限必要な問題を抜粋したものですが、[Introduction_to_Pandas](./11_Introduction_to_Pandas.ipynb) に掲載されていない機能が必要な問題もあります。
初めて触るライブラリを調べながら使うというのはよくある光景です。この課題では皆さんにもそれに挑戦していただきます。  
ヒントとして検索キーワードなどを載せておくので、自力で調べながら解いてみましょう。  



## 必要モジュールのインポート

この問題で使うモジュールをインポートします．

In [35]:
import pandas as pd
import numpy as np

## データの読み込み

In [36]:
df_customer = pd.read_csv('https://raw.githubusercontent.com/The-Japan-DataScientist-Society/100knocks-preprocess/master/docker/work/data/customer.csv')
df_product = pd.read_csv('https://raw.githubusercontent.com/The-Japan-DataScientist-Society/100knocks-preprocess/master/docker/work/data/product.csv')
df_receipt = pd.read_csv('https://raw.githubusercontent.com/The-Japan-DataScientist-Society/100knocks-preprocess/master/docker/work/data/receipt.csv')

---
## 問1. 条件抽出
> P-006: レシート明細データフレーム「df_receipt」から売上日（sales_ymd）、顧客ID（customer_id）、商品コード（product_cd）、売上数量（quantity）、売上金額（amount）の順に列を指定し、以下の条件を満たすデータを抽出せよ。
> - 顧客ID（customer_id）が"CS018205000001"
> - 売上金額（amount）が1,000以上または売上数量（quantity）が5以上

In [37]:
df_receipt[
    (df_receipt['customer_id'] == 'CS018205000001') &
    ((df_receipt['amount'] >= 1000) | (df_receipt['quantity'] >= 5))
]

,sales_ymd,sales_epoch,store_cd,receipt_no,receipt_sub_no,customer_id,product_cd,quantity,amount
36,20180911,1536624000,S13018,1122,2,CS018205000001,P071401012,1,2200
9843,20180414,1523664000,S13018,1142,2,CS018205000001,P060104007,6,600
21110,20170614,1497398400,S13018,1112,2,CS018205000001,P050206001,5,990
68117,20190226,1551139200,S13018,1132,1,CS018205000001,P071401020,1,2200
72254,20180911,1536624000,S13018,1122,1,CS018205000001,P071401005,1,1100


---
## 問2. ソート
> P-18: 顧客データフレーム（df_customer）を生年月日（birth_day）で若い順にソートし、先頭5件を全項目表示せよ。

In [38]:
df_customer.sort_values(by='birth_day', ascending=False).head()

,customer_id,customer_name,gender_cd,gender,birth_day,age,postal_cd,address,application_store_cd,application_date,status_cd
15639,CS035114000004,大村 美里,1,女性,2007-11-25,11,156-0053,東京都世田谷区桜**********,S13035,20150619,6-20091205-6
7468,CS022103000002,福山 はじめ,9,不明,2007-10-02,11,249-0006,神奈川県逗子市逗子**********,S14022,20160909,0-00000000-0
10745,CS002113000009,柴田 真悠子,1,女性,2007-09-17,11,184-0014,東京都小金井市貫井南町**********,S13002,20160304,0-00000000-0
19811,CS004115000014,松井 京子,1,女性,2007-08-09,11,165-0031,東京都中野区上鷺宮**********,S13004,20161120,1-20081231-1
7039,CS002114000010,山内 遥,1,女性,2007-06-03,11,184-0015,東京都小金井市貫井北町**********,S13002,20160920,6-20100510-1


---
## 問3. 全件数
> P-021: レシート明細データフレーム（df_receipt）に対し、件数をカウントせよ。

In [39]:
total_count = df_receipt.shape[0]
print(total_count)

104681


## 問4. ユニーク件数
> P-022: レシート明細データフレーム（df_receipt）の顧客ID（customer_id）に対し、ユニーク件数をカウントせよ。

In [40]:
unique_customer_count = df_receipt['customer_id'].nunique()
print(unique_customer_count)

8307


<details>
<summary>ヒント</summary>
「ユニーク」というのはそのまま検索に使える単語です。  
</details>

---
## 問5. 〇〇ごとに集計
> P-035: レシート明細データフレーム（df_receipt）に対し、顧客ID（customer_id）ごとに売上金額（amount）を合計して全顧客の平均を求め、平均以上に買い物をしている顧客を抽出せよ。ただし、顧客IDが"Z"から始まるのものは非会員を表すため、除外して計算すること。なお、データは先頭5件だけ表示せよ。

会員のみを抽出する方法は、例えば以下の2通りの方法があります。

In [61]:
df_receipt_only_member = df_receipt[~df_receipt["customer_id"].str.startswith("Z")]
df_receipt_only_member = df_receipt.query("not customer_id.str.startswith('Z')", engine="python")

In [32]:
# 顧客IDごとに売上金額を合計
amount_customer = df_receipt_only_member.groupby('customer_id')['amount'].sum()

# 全顧客の平均を計算
average = amount_customer.mean()

# 平均以上に買い物をしている顧客を抽出
high_spending_customers = amount_customer[amount_customer >= average]

# 結果を先頭5件表示
result = high_spending_customers.head()

print(result)

customer_id
CS001115000010    3044
CS001205000006    3337
CS001214000009    4685
CS001214000017    4132
CS001214000052    5639
Name: amount, dtype: int64


<details>
<summary>ヒント1</summary>
「pandas 要素ごと 集計」 などで今回使える機能に関する記事が見つかります。
</details>

<details>
<summary>ヒント2</summary>
メソッド名は "groupby" です。
</details>

---
## 問6. DataFrameの結合
> P-038: 顧客データフレーム（df_customer）とレシート明細データフレーム（df_receipt）から、各顧客ごとの売上金額合計を求めよ。ただし、買い物の実績がない顧客については売上金額を0として表示させること。また、顧客は性別コード（gender_cd）が女性（1）であるものを対象とし、非会員（顧客IDが'Z'から始まるもの）は除外すること。なお、結果は先頭5件だけ表示せよ。

In [48]:
df_customer_only_member = df_customer[~df_customer["customer_id"].str.startswith("Z")]
df_customer_only_member = df_customer.query("not customer_id.str.startswith('Z')", engine="python")

In [79]:
# 女性の顧客を対象
df_customer_female = df_customer_only_member[df_customer_only_member["gender_cd"] == 1]

# 顧客ごとの売上金額合計を計算
df_sales_sum = df_receipt.groupby('customer_id')['amount'].sum().reset_index()

# すべての女性顧客を対象にして売上金額を0として表示
df_sales_sum_full = pd.merge(df_customer_female[['customer_id']], df_sales_sum, on='customer_id', how='left').fillna(0)

df_sales_sum_full = df_sales_sum_full[['customer_id', 'amount']]
df_sales_sum_full.head()

,customer_id,amount
0,CS021313000114,0.0
1,CS031415000172,5088.0
2,CS028811000001,0.0
3,CS001215000145,875.0
4,CS015414000103,3122.0


<details>
<summary>ヒント1</summary>
タイトル通り 「pandas DataFrame 結合」などと調べれば必要な機能に関する記事が見つかります。  
</details>


<details>
<summary>ヒント2</summary>
"merge", "join"という似たメソッドがあります。  
今回の場合"merge"が便利でしょう。
</details>

---
## 問7. 時系列データ
> P-046: 顧客データフレーム（df_customer）の申し込み日（application_date）はYYYYMMD形式の文字列型でデータを保有している。これを日付型（dateやdatetime）に変換し、顧客ID（customer_id）とともに抽出せよ。なお、データは先頭5件を表示せよ。

In [57]:
# application_dateを日付型に変換
df_customer['application_date'] = pd.to_datetime(df_customer['application_date'], format='%Y%m%d')

# 必要な列を抽出し、先頭5件を表示
df_customer[['customer_id', 'application_date']].head()

,customer_id,application_date
0,CS021313000114,2015-09-05
1,CS037613000071,2015-04-14
2,CS031415000172,2015-05-29
3,CS028811000001,2016-01-15
4,CS001215000145,2017-06-05


<details>
<summary>ヒント1</summary>
「pandas datetime」などで該当の機能が見つかるかと思います。
</details>


<details>
<summary>ヒント2</summary>
"pd.to_datetime"というメソッドが使えるでしょう。このメソッドを適用する際ですが、for文を使わずに実装しましょう。

---
## 問8. 関数
> P-061: レシート明細データフレーム（df_receipt）の売上金額（amount）を顧客ID（customer_id）ごとに合計し、合計した売上金額を常用対数化（底=10）して顧客ID、売上金額合計とともに表示せよ。ただし、顧客IDが"Z"から始まるのものは非会員を表すため、除外して計算すること。なお、結果は先頭5件を表示せよ。

In [65]:
# 顧客IDごとに売上金額を合計
df_amount_sum = df_receipt_only_member.groupby('customer_id')['amount'].sum().reset_index()

# 売上金額合計を常用対数化
df_amount_sum['log_amount_sum'] = np.log10(df_amount_sum['amount'])

# 必要な列を抽出し、先頭5件を表示
df_amount_sum[['customer_id', 'amount', 'log_amount_sum']].head(5)

,customer_id,amount,log_amount_sum
0,CS001113000004,1298,3.113275
1,CS001114000005,626,2.796574
2,CS001115000010,3044,3.483445
3,CS001205000004,1988,3.298416
4,CS001205000006,3337,3.523356


---
## 問9. 欠損数
> P-079: 商品データフレーム（df_product）の各項目に対し、欠損数を確認せよ。

In [66]:
df_product.isnull().sum()

product_cd            0
category_major_cd     0
category_medium_cd    0
category_small_cd     0
unit_price            7
unit_cost             7
dtype: int64

---
## 問10. 欠損値の除去
> P-080: 商品データフレーム（df_product）のいずれかの項目に欠損が発生しているレコードを全て削除した新たなdf_product_1を作成せよ。なお、削除前後の件数を表示させ、前設問で確認した件数だけ減少していることも確認すること。

In [71]:
# 削除前の件数を取得
count_before = df_product.shape[0]

# 欠損値を含むレコードを削除
df_product_1 = df_product.dropna()

# 削除後の件数を取得
count_after = df_product_1.shape[0]

# 件数の表示
print(f"削除前の件数: {count_before}")
print(f"削除後の件数: {count_after}")

# 削除前後の件数の差を表示
print(f"削除された件数: {count_before - count_after}")

削除前の件数: 10030
削除後の件数: 10023
削除された件数: 7


In [16]:
len(df_product), len(df_product_1)

(10030, 10023)

---
## 問11. 欠損値の穴埋め
> P-081: 単価（unit_price）と原価（unit_cost）の欠損値について、それぞれの平均値で補完した新たなdf_product_2を作成せよ。なお、平均値について1円未満は四捨五入とせよ。補完実施後、各項目について欠損が生じていないことも確認すること。

In [81]:
# それぞれの列の平均値を計算（1円未満は四捨五入）
unit_price_mean = round(df_product_2['unit_price'].mean())
unit_cost_mean = round(df_product_2['unit_cost'].mean())

# 平均値で欠損値を補完
df_product_2['unit_price'].fillna(unit_price_mean, inplace=True)
df_product_2['unit_cost'].fillna(unit_cost_mean, inplace=True)

# 補完実施後、欠損が生じていないことを確認
no_missing_values = df_product_2.isnull().sum().sum() == 0

# 結果を表示
print("補完後のデータフレーム:")
print(df_product_2)
print(f"\n欠損値がないかの確認: {no_missing_values}")

補完後のデータフレーム:
       product_cd  category_major_cd  category_medium_cd  category_small_cd  \
0      P040101001                  4                 401              40101   
1      P040101002                  4                 401              40101   
2      P040101003                  4                 401              40101   
3      P040101004                  4                 401              40101   
4      P040101005                  4                 401              40101   
...           ...                ...                 ...                ...   
10025  P091503001                  9                 915              91503   
10026  P091503002                  9                 915              91503   
10027  P091503003                  9                 915              91503   
10028  P091503004                  9                 915              91503   
10029  P091503005                  9                 915              91503   

       unit_price  unit_cost  
0      

/var/folders/mp/5nk3w81d2pb5jy03j_yx72hw0000gn/T/ipykernel_18073/4293096857.py:6: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df_product_2['unit_price'].fillna(unit_price_mean, inplace=True)
/var/folders/mp/5nk3w81d2pb5jy03j_yx72hw0000gn/T/ipykernel_18073/4293096857.py:7: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are set

### 余談
ChatGPTやBing AIに聞けば大抵のことは教えてくれます。  
何回か入力文章を吟味しないといけないこともありますが、知らないことを調べる場合は自分で検索するよりも早いです。  
ただ、ChatGPTなどは嘘をつく場合があるので、自分でソースを参照する姿勢は必要です。  

これはBingAIの回答例です。  

![BingAIの回答例](./imgs/pandas/BingAI.png)